<a href="https://colab.research.google.com/github/sunilthapa-in/sunilthapa-in/blob/main/SNNFINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install sinabs

INFO: pip is looking at multiple versions of nirtorch to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.9/131.9 kB 8.6 MB/s eta 0:00:00


In [2]:
pip install samna

In [ ]:
pip install sinabs.exodus

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.5/52.5 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
pip install torchmetrics

In [ ]:
import torch
import sinabs
import sinabs.layers as sl
import sinabs.exodus.layers as sel
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import torchmetrics
from sinabs.exodus.conversion import exodus_to_sinabs
from google.colab import drive
import os
import pickle
from sklearn.model_selection import train_test_split

# Mount Google Drive
drive.mount('/content/drive')

# Define sensor size and number of time bins
sensor_size = (128, 128)
num_time_bins = 30

# Custom Dataset class for loading tensor event data
class TensorEventDataset(torch.utils.data.Dataset):
    def __init__(self, data_files):
        self.data_files = data_files

    def __len__(self):
        return len(self.data_files)

    def __getitem__(self, idx):
        with open(self.data_files[idx], 'rb') as file:
            structured_tensor = pickle.load(file)
        # Extract the label from the filename (digit between underscores)
        filename = os.path.basename(self.data_files[idx])
        label = int(filename.split('_')[1])
        # Ensure the tensor has the correct shape [num_time_bins, channels, height, width]
        structured_tensor = structured_tensor.permute(1, 0, 2, 3)
        return structured_tensor, label

# Load dataset
data_dir = '/content/drive/My Drive/Final_Data/Centre/MG400_tapping20240711_134233/tensor_events_2_interchanged'
data_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith('_structured_tensor.pickle')]
data_files.sort()

# Print the total number of files
print(f"Total number of files: {len(data_files)}")

# Split data into train and test sets
train_files, test_files = train_test_split(data_files, test_size=0.2, random_state=42)

# Print the number of training and testing files
print(f"Number of training files: {len(train_files)}")
print(f"Number of testing files: {len(test_files)}")

# Create datasets and dataloaders
trainset = TensorEventDataset(train_files)
testset = TensorEventDataset(test_files)

batch_size = 4

trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, num_workers=4)

# Define the SNN model

backend = sel  # Sinabs EXODUS

model = nn.Sequential(
    sl.FlattenTime(),
    nn.Conv2d(2, 8, kernel_size=3, padding=1, bias=False),
    backend.IAFSqueeze(batch_size=batch_size, min_v_mem=-1),
    sl.SumPool2d(2),
    nn.Conv2d(8, 16, kernel_size=3, padding=1, bias=False),
    backend.IAFSqueeze(batch_size=batch_size, min_v_mem=-1),
    sl.SumPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1, bias=False),
    backend.IAFSqueeze(batch_size=batch_size, min_v_mem=-1),
    sl.SumPool2d(2),
    nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
    backend.IAFSqueeze(batch_size=batch_size, min_v_mem=-1),
    sl.SumPool2d(2),
    nn.Conv2d(64, 10, kernel_size=2, padding=0, bias=False),
    backend.IAFSqueeze(batch_size=batch_size, min_v_mem=-1),
    nn.Flatten(),
    sl.UnflattenTime(batch_size=batch_size),
).cuda()

# Print the number of layers in the model
layer_count = sum(1 for _ in model)
print(f"Number of layers in the model: {layer_count}")

# Define the optimizer and loss function
n_epochs = 1
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.functional.cross_entropy

# Training loop
for epoch in range(n_epochs):
    losses = []
    for data, targets in tqdm(trainloader):
        data, targets = data.cuda(), targets.clone().detach().cuda()  # Ensure targets are tensors
        sinabs.reset_states(model)
        optimizer.zero_grad()

        # Reshape the data to ensure it has the desired shape [16, 30, 2, 128, 128]
        data = data.view(batch_size, num_time_bins, 2, sensor_size[0], sensor_size[1])

        # Calculate y_hat within the loop for each batch
        y_hat = model(data)
        num_time_steps = y_hat.shape[1]
        pred = y_hat.view(batch_size, num_time_steps, -1).sum(1)  # Sum over time steps

        loss = crit(pred, targets)
        loss.backward()
        losses.append(loss)
        optimizer.step()
    print(f"Loss: {torch.stack(losses).mean()}")

# Evaluate the model
acc = torchmetrics.Accuracy('multiclass', num_classes=490).cuda()
model.eval()

for data, targets in tqdm(testloader):
    data, targets = data.cuda(), targets.clone().detach().cuda()  # Ensure targets are tensors
    sinabs.reset_states(model)
    with torch.no_grad():

        data = data.view(batch_size, num_time_bins, 2, sensor_size[0], sensor_size[1])
        y_hat = model(data)
    pred = y_hat.view(batch_size, num_time_bins, -1).sum(1)  # Ensure correct shape before summing
    acc(pred, targets)

print(f"Test accuracy: {100 * acc.compute():.2f}%")

# Convert to Sinabs model and save
sinabs_model = exodus_to_sinabs(model)
torch.save(sinabs_model, "prototype_model.pth")


In [ ]:
[4, 30, 2, 128, 128].numel


In [ ]:
import torch
import sinabs
import sinabs.layers as sl
import sinabs.exodus.layers as sel
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import torchmetrics
from sinabs.exodus.conversion import exodus_to_sinabs
from google.colab import drive
import os
import pickle
from sklearn.model_selection import train_test_split

# Mount Google Drive
drive.mount('/content/drive')

# Define sensor size and number of time bins
sensor_size = (128, 128)
num_time_bins = 30
batch_size = 16

# Custom Dataset class for loading tensor event data
class TensorEventDataset(torch.utils.data.Dataset):
    def __init__(self, data_files):
        self.data_files = data_files

    def __len__(self):
        return len(self.data_files)

    def __getitem__(self, idx):
        with open(self.data_files[idx], 'rb') as file:
            structured_tensor = pickle.load(file)
        # Extract the label from the filename (digit between underscores)
        filename = os.path.basename(self.data_files[idx])
        label = int(filename.split('_')[1])
        return structured_tensor, label

# Load dataset
data_dir = '/content/drive/My Drive/Final_Data/Centre/MG400_tapping20240711_134233/tensor_events_2_interchanged'
data_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith('_structured_tensor.pickle')]
data_files.sort()

# Print the total number of files
print(f"Total number of files: {len(data_files)}")

# Split data into train and test sets
train_files, test_files = train_test_split(data_files, test_size=0.2, random_state=42)

# Print the number of training and testing files
print(f"Number of training files: {len(train_files)}")
print(f"Number of testing files: {len(test_files)}")

# Create datasets and dataloaders
trainset = TensorEventDataset(train_files)
testset = TensorEventDataset(test_files)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, num_workers=4)

# Define the SNN model
backend = sel  # Sinabs EXODUS

model = nn.Sequential(
    sl.FlattenTime(),
    nn.Conv2d(2, 8, kernel_size=3, padding=1, bias=False),
    backend.IAFSqueeze(batch_size=batch_size, min_v_mem=-1),
    sl.SumPool2d(2),
    nn.Conv2d(8, 16, kernel_size=3, padding=1, bias=False),
    backend.IAFSqueeze(batch_size=batch_size, min_v_mem=-1),
    sl.SumPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1, bias=False),
    backend.IAFSqueeze(batch_size=batch_size, min_v_mem=-1),
    sl.SumPool2d(2),
    nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
    backend.IAFSqueeze(batch_size=batch_size, min_v_mem=-1),
    sl.SumPool2d(2),
    nn.Conv2d(64, 10, kernel_size=2, padding=0, bias=False),
    backend.IAFSqueeze(batch_size=batch_size, min_v_mem=-1),
    nn.Flatten(),
    sl.UnflattenTime(batch_size=batch_size),
).cuda()

# Print the number of layers in the model
layer_count = sum(1 for _ in model)
print(f"Number of layers in the model: {layer_count}")

# Define the optimizer and loss function
n_epochs = 1
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.functional.cross_entropy

# Training loop
for epoch in range(n_epochs):
    losses = []
    for data, targets in tqdm(trainloader):
        data, targets = data.cuda(), targets.clone().detach().cuda()  # Ensure targets are tensors
        sinabs.reset_states(model)
        optimizer.zero_grad()

        # Calculate y_hat within the loop for each batch
        y_hat = model(data)
        num_time_steps = y_hat.shape[1]
        pred = y_hat.view(batch_size, num_time_steps, -1).sum(1)  # Sum over time steps

        loss = crit(pred, targets)
        loss.backward()
        losses.append(loss)
        optimizer.step()
    print(f"Loss: {torch.stack(losses).mean()}")

# Evaluate the model
acc = torchmetrics.Accuracy('multiclass', num_classes=10).cuda()
model.eval()

for data, targets in tqdm(testloader):
    data, targets = data.cuda(), targets.clone().detach().cuda()  # Ensure targets are tensors
    sinabs.reset_states(model)
    with torch.no_grad():
        y_hat = model(data)
    pred = y_hat.view(batch_size, num_time_bins, -1).sum(1)  # Ensure correct shape before summing
    acc(pred, targets)

print(f"Test accuracy: {100 * acc.compute():.2f}%")

# Convert to Sinabs model and save
sinabs_model = exodus_to_sinabs(model)
torch.save(sinabs_model, "prototype_model.pth")


In [ ]:
import torch
import sinabs
import sinabs.layers as sl
import sinabs.exodus.layers as sel
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import torchmetrics
from sinabs.exodus.conversion import exodus_to_sinabs
from google.colab import drive
import os
import pickle
from sklearn.model_selection import train_test_split

# Mount Google Drive
drive.mount('/content/drive')

# Define sensor size and number of time bins
sensor_size = (128, 128)
num_time_bins = 30
batch_size = 16

# Custom Dataset class for loading tensor event data
class TensorEventDataset(torch.utils.data.Dataset):
    def __init__(self, data_files):
        self.data_files = data_files

    def __len__(self):
        return len(self.data_files)

    def __getitem__(self, idx):
        with open(self.data_files[idx], 'rb') as file:
            structured_tensor = pickle.load(file)
        # Extract the label from the filename (digit between underscores)
        filename = os.path.basename(self.data_files[idx])
        label = int(filename.split('_')[1])
        return structured_tensor, label

# Load dataset
data_dir = '/content/drive/My Drive/Final_Data/Centre/MG400_tapping20240711_134233/tensor_events_2_interchanged'
data_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith('_structured_tensor.pickle')]
data_files.sort()

# Print the total number of files
print(f"Total number of files: {len(data_files)}")

# Split data into train and test sets
train_files, test_files = train_test_split(data_files, test_size=0.2, random_state=42)

# Print the number of training and testing files
print(f"Number of training files: {len(train_files)}")
print(f"Number of testing files: {len(test_files)}")

# Create datasets and dataloaders
trainset = TensorEventDataset(train_files)
testset = TensorEventDataset(test_files)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, num_workers=4)

# Define the SNN model
backend = sel  # Sinabs EXODUS

model = nn.Sequential(
    nn.Conv2d(2, 8, kernel_size=3, padding=1, bias=False),
    backend.IAFSqueeze(batch_size=batch_size, min_v_mem=-1),
    sl.SumPool2d(2),
    nn.Conv2d(8, 16, kernel_size=3, padding=1, bias=False),
    backend.IAFSqueeze(batch_size=batch_size, min_v_mem=-1),
    sl.SumPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1, bias=False),
    backend.IAFSqueeze(batch_size=batch_size, min_v_mem=-1),
    sl.SumPool2d(2),
    nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
    backend.IAFSqueeze(batch_size=batch_size, min_v_mem=-1),
    sl.SumPool2d(2),
    nn.Conv2d(64, 10, kernel_size=2, padding=0, bias=False),
    backend.IAFSqueeze(batch_size=batch_size, min_v_mem=-1),
    nn.Flatten(),
    sl.UnflattenTime(batch_size=batch_size),
).cuda()

# Print the number of layers in the model
layer_count = sum(1 for _ in model)
print(f"Number of layers in the model: {layer_count}")

# Define the optimizer and loss function
n_epochs = 1
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.functional.cross_entropy

# Training loop
for epoch in range(n_epochs):
    losses = []
    for data, targets in tqdm(trainloader):
        data, targets = data.cuda(), targets.clone().detach().cuda()  # Ensure targets are tensors
        sinabs.reset_states(model)
        optimizer.zero_grad()

        # Print the shape of the data before reshaping
        print(f"Original data shape: {data.shape}")

        # Reshape the data to merge the time dimension with the batch size
        data = data.view(-1, 2, sensor_size[0], sensor_size[1])

        # Print the shape of the data after reshaping
        print(f"Reshaped data shape: {data.shape}")

        # Calculate y_hat within the loop for each batch
        y_hat = model(data)
        y_hat = y_hat.view(batch_size, num_time_bins, -1)  # Reshape back to [batch_size, num_time_bins, features]
        pred = y_hat.sum(1)  # Sum over time steps

        loss = crit(pred, targets)
        loss.backward()
        losses.append(loss)
        optimizer.step()
    print(f"Loss: {torch.stack(losses).mean()}")

# Evaluate the model
acc = torchmetrics.Accuracy('multiclass', num_classes=10).cuda()
model.eval()

for data, targets in tqdm(testloader):
    data, targets = data.cuda(), targets.clone().detach().cuda()  # Ensure targets are tensors
    sinabs.reset_states(model)
    with torch.no_grad():
        # Print the shape of the data before reshaping
        print(f"Original data shape: {data.shape}")

        # Reshape the data to merge the time dimension with the batch size
        data = data.view(-1, 2, sensor_size[0], sensor_size[1])

        # Print the shape of the data after reshaping
        print(f"Reshaped data shape: {data.shape}")

        y_hat = model(data)
        y_hat = y_hat.view(batch_size, num_time_bins, -1)  # Reshape back to [batch_size, num_time_bins, features]
        pred = y_hat.sum(1)  # Sum over time steps
    acc(pred, targets)

print(f"Test accuracy: {100 * acc.compute():.2f}%")

# Convert to Sinabs model and save
sinabs_model = exodus_to_sinabs(model)
torch.save(sinabs_model, "prototype_model.pth")
